# tp-23 アンサンブル v3（追い切りラップ TL_A を全メンバーに足して再学習）

tp-19 で採用した追い切りの時計・加速度の特徴量（TL_A、勝率 −0.00136）を、アンサンブル v2 の 7 メンバー全員の特徴量に足して再学習し、v2 と同じ手順で合成し直す。

| メンバー | 特徴量（v2 と同じ + TL_A） | パラメータ |
|----------|---------------------------|------------|
| lgb_extras_ped | base+S+C1 + 追い切り評価・厩舎コメント + 血統×条件 | LightGBM（調整後） |
| xgb_extras_ped | 同上 | XGBoost（調整後） |
| nn_extras_ped | 同上 | NN（調整後） |
| lgb_extras | base+S+C1 + 追い切り評価・厩舎コメント | LightGBM（調整後） |
| lgb_softmax | base+S+C1 | レース内 softmax |
| linear | base+S+C1 | 線形（14 epoch） |
| nn_s | base+S+C1 | NN (128,64)、14 epoch |

合成は v2 と同じ（勝率: logit の重み付き平均、連対・複勝: 前進選択）。評価は leave-one-year-out の OOF（valid 2022〜2024）で、**v2 に対する差**をレース単位の paired bootstrap で測る（採用基準 = 上限 &lt; 0、効果 ≥ 0.0002、連対・複勝が 0.0002 を超えて悪化しない）。**test 2025 は使わない**（採用が決まれば tp-06c で一度だけ）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ensemble as en, tp05_data, win_models as wm, zoo, zoo_runner, feature_bakeoff as fb
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "ensemble_v3"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP23_THREADS", detect_compute(reserve_cores=6).threads))
SMOKE = os.environ.get("TP23_SMOKE") == "1"
SEEDS = [0] if SMOKE else [0, 1, 2]
mlflow.set_experiment("keiba-tp-ensemble")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X1 = df[D["FEATS"]["base+S+C1"]]
TL = pd.read_parquet(WORK / "training_laps_b/cand_TL_A.parquet")
TLm = df[["race_id", "horse_id"]].merge(TL, on=["race_id", "horse_id"], how="left")
TLm.index = df.index
TL_COLS = [c for c in TLm.columns if c not in ("race_id", "horse_id")]
TLX = TLm[TL_COLS].add_prefix("tl_")
print("行", len(df), "| TL_A 列", len(TL_COLS), "| TL_A の欠損率", round(float(TLX.isna().mean().mean()), 3))
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP, XP, NP = zoo.tuned_params("lgb", best["lgb"]["params"]), zoo.tuned_params("xgb", best["xgb"]["params"]), zoo.tuned_params("nn", best["nn"]["params"])
if SMOKE:
    folds = folds[:2]

## 7 メンバーを TL_A つきで再学習（OOF）

In [ ]:
X_ep = pd.concat([X1, EP], axis=1)                              # extras + ped
X_ex = pd.concat([X1, EP[EXTRAS]], axis=1)                      # extras のみ
def with_tl(X):
    return pd.concat([X, TLX], axis=1)

MEMBERS = {                                                      # 名前: (recipe, params, 特徴量, v2 の対応メンバー)
    "lgb_extras_ped": ("lgb_binary", LP, X_ep),
    "xgb_extras_ped": ("xgb_binary", XP, X_ep),
    "nn_extras_ped": ("nn", NP, X_ep),
    "lgb_extras": ("lgb_binary", LP, X_ex),
    "lgb_softmax": ("lgb_softmax", {}, X1),
    "linear": ("linear", {"epochs": 14}, X1),
    "nn_s": ("nn", {"hidden": (128, 64), "epochs": 14}, X1),
}
if SMOKE:
    MEMBERS = {k: v for k, v in MEMBERS.items() if k in ("lgb_extras", "linear")}
key = ["race_id", "horse_id"]
OOF3, RES = {}, []
for name, (recipe, params, X) in MEMBERS.items():
    t0 = time.time()
    res, oof = zoo_runner.run_oof(f"{name}_tl3", recipe, with_tl(X), meta, folds, SEEDS, THREADS, params=dict(params), out_dir=OOF_DIR if not SMOKE else None)
    OOF3[name] = oof
    RES.append(res.assign(member=name))
    old = pd.read_parquet(OOF_DIR / f"{name}.parquet") if (OOF_DIR / f"{name}.parquet").exists() else None
    msg = f"{name}: win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({time.time() - t0:.0f}s)"
    print(msg, flush=True)
    gc.collect()
pd.concat(RES).to_csv(OUT / "members_tl3.csv", index=False)

## v2 と同じ手順で合成し、v2 と比べる

In [ ]:
names = list(OOF3)
base = meta[key + ["finish_position", "year"]].copy()
for n, o in OOF3.items():
    o = o[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in ("win", "top2", "top3")})
    base = base.merge(o, on=key, how="inner")
v2 = pd.read_parquet(OOF_DIR / "ensemble_v2.parquet")[key + ["p_win", "p_top2", "p_top3"]]
race, years, fin = base["race_id"].to_numpy(), base["year"].to_numpy(), base["finish_position"].to_numpy()
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
Y = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
METHOD = {"win": "weighted_logit", "top2": "greedy", "top3": "greedy"}            # v2 と同じ
ENS = base[key + ["year"]].copy()
SPEC = {}
for t in TOT:
    P = np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1)
    llv, pred = en.loyo_eval(METHOD[t], P, years, race, Y[t], TOT[t])
    ENS[f"p_{t}"] = pred
    st = en.fit_method(METHOD[t], P, en.logit(P), Y[t])                              # 全 OOF で当てはめた重み（参考）
    SPEC[t] = {"method": METHOD[t], "weights": [float(x) for x in st]}
    print(t, "LOYO LogLoss", round(llv, 5), "| 重み", dict(zip(names, np.round(st, 3))))
ENS.to_parquet(OOF_DIR / "ensemble_v3.parquet" if not SMOKE else OUT / "ensemble_v3_smoke.parquet", index=False)
fin_map = meta.set_index(key)["finish_position"]
cmp_ = fb.compare_oof(v2.rename(columns={}), ENS[key + ["p_win", "p_top2", "p_top3"]], fin_map, "ensemble v3 vs v2")
print({k: (round(v, 5) if isinstance(v, float) else v) for k, v in cmp_.items()})

In [ ]:
(OUT / "ensemble_spec.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "models": names, "targets": SPEC, "tl_cols": TL_COLS}, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "vs_v2": cmp_, "members": pd.concat(RES).groupby("member")[["logloss_win", "logloss_top2", "logloss_top3"]].mean().round(6).reset_index().to_dict("records")}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
if not SMOKE:
    with mlflow.start_run(run_name="tp-23-ensemble-v3", tags={"notebook": "tp-23", "git_commit": GIT_COMMIT}):
        mlflow.log_metrics({"delta_win_vs_v2": float(cmp_["Δwin"]), "delta_top2_vs_v2": float(cmp_["Δtop2"]), "delta_top3_vs_v2": float(cmp_["Δtop3"]), "win_v3": float(cmp_["win_new"])})
        mlflow.log_artifact(str(OUT / "report.json"), "ensemble_v3"); mlflow.log_artifact(str(OUT / "ensemble_spec.json"), "ensemble_v3")